This notebook demonstrates how to do custom object detection using your local webcam (computer camera).

**Prerequisites**: 
- Python 3.
- A Google account is recommended to access Google Colab's GPU for training.
- A Roboflow account is recommended for annotation, though you may use other annotation tools as you wish.

**Note on Google Colab**: A local webcam cannot be accessed from Google Colab. Capture and testing of the live webcam feed must be done with a local Python kernel on your computer. Training the model may still be done on Colab (recommended) to use a GPU.

Unfortunately, as the Google Colab extension in VS Code currently does not support access to Google Drive, and also cannot access local files, you will need to use Google Colab in the browser.

In [ ]:
%pip install ultralytics opencv-python

In [8]:
import cv2
import numpy as np
import time

from IPython.display import clear_output

from ultralytics import YOLO

# Use local webcam instead of UGOT camera
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam (device 0).")

The following few cells allow you to see how the pretrained YOLO model sees objects before you train the model on your own data. This must be done locally because Google Colab cannot access a local webcam.

Run these cells in a local Python kernel to view the live webcam feed and test detections.

In [2]:
# Load pretrained model from YOLO 
model = YOLO("yolo11n.pt")

In [ ]:
# Helper: Draw bounding boxes
def draw_detections(frame, results):
    for r in results:
        boxes = r.boxes  # bounding boxes

        for box in boxes:
            # xyxy format: [x1, y1, x2, y2]
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)

            # Confidence & label
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            label = r.names[cls_id]

            # Draw rectangle
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            # Label text
            text = f"{label} {conf:.2f}"
            cv2.putText(frame, text, (x1, y1 - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6,
                        (0, 255, 0), 2)
    return frame

In [4]:
# Visualize bounding boxes with live video feed
while True:
    ret, img = cap.read()
    if not ret:
        continue

    # Run YOLO detection
    results = model(img, verbose=False)

    # Draw output
    output = draw_detections(img, results)

    # Show
    cv2.imshow("YOLO Detection", output)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


To train the YOLO model using our own images, we need to first capture some images of the object we want to detect.
The following cell will automatically capture an image from your webcam at regular intervals.
Here are some tips for good training data:
- Capture at least 50 images. The more, the better, but remember you will also need to annotate them.
- Move the camera and objects around so that you get images when the object is near/far, under different lighting conditions, partially obscured, multiple objects in the same picture, etc. Think about what kind of conditions the system might encounter during the competition.

In [5]:
# Save images from UGOT video feed at regular time intervals
import cv2
import numpy as np
import time
import os

SAVE_DIR = "captured_demo" # changed folder for demo purposes
os.makedirs(SAVE_DIR, exist_ok=True)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("Could not open webcam (device 0).")

counter = 0    # with multiple runs, change this to one after the last captured image name to avoid overwriting images
interval = 1   # seconds between captures

print("Auto-capturing images. Press 'q' to stop.")

last_time = time.time()

try:
    while True:
        ret, img = cap.read()
        if not ret:
            continue

        cv2.imshow("Webcam", img)

        # Auto-save
        if time.time() - last_time >= interval:
            filename = f"{SAVE_DIR}/img_{counter:04d}.jpg"
            cv2.imwrite(filename, img)
            print(f"Saved: {filename}")
            counter += 1
            last_time = time.time()

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Done.")


Auto-capturing images. Press 'q' to stop.
Saved: captured_demo/img_0000.jpg
Saved: captured_demo/img_0001.jpg
Saved: captured_demo/img_0002.jpg
Saved: captured_demo/img_0003.jpg
Saved: captured_demo/img_0004.jpg
Saved: captured_demo/img_0005.jpg
Saved: captured_demo/img_0006.jpg
Saved: captured_demo/img_0007.jpg
Done.


After capturing the images, we need to annotate them with the bounding boxes and class labels. Create an account at [app.roboflow.com](https://app.roboflow.com/).
1. Create a new project, select `Traditional` (not Rapid) tool and `Object Detection` project type, and upload your images.
2. Use the tools to annotate your images (label each object with your classes). If you have created a new account, you should be able to use the built-in AI annotation tool to speed up the process as part of your free trial. Ensure that the bounding boxes are tight.
3. You may apply augmentations in the Dataset tab if you wish. However, augmented images can be generated on the fly by instead modifying the `augments` parameter in `model.train` below.
4. Select a train/test/val split when creating a version of your dataset to download. You must have some images in each. The default 70/20/10 split should do.
5. Under the Versions tab, download the dataset in the YOLOv11 format (select "Download zip to computer", download, and unzip).
6. Upload the folder to your Google Drive.

Now, we need to train the model. Note that since this overwrites the previous model classes, only your new classes that you have annotated will be detected, although we use YOLOv11 as the base model.

1. Open this notebook in Google Colab. In the top right corner of the browser webpage, select "Change runtime type" and select an available GPU (usually T4). If you select CPU, or simply try to run this cell on your computer, the training will take much longer.
2. Change the path in `model.train` below to the correct path to `data.yaml` in your Drive folder. Change the paths of train / val / test in `data.yaml` as well. You can copy the path from the file explorer on the left.

After training, **remember to download the model** from `runs/detect/trainx/weights` where `x` is the $x$th time you have trained a model in the current runtime. The weights will be deleted once the runtime disconnects. You probably only need `best.pt`.

In [ ]:
# Allow Google Colab to access your Google Drive files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Use pretrained model from YOLO
model = YOLO("yolo11n.pt")

# Train on new roboflow data
model.train(data="/content/drive/MyDrive/roboflow_coffee/data.yaml", epochs=50, imgsz=512)
# model.train(data="./roboflow_bad/data.yaml", epochs=50, imgsz=512)

Now test your model with the new objects (remember to switch back to local and run imports and definitions):

In [9]:
trained = YOLO("best_coffee.pt")

# Use the same webcam capture used earlier (or reopen if running cells independently)
try:
    cap
except NameError:
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        raise RuntimeError("Could not open webcam (device 0).")

while True:
    ret, img = cap.read()
    if not ret:
        continue

    # Run YOLO detection
    results = trained(img, verbose=False)

    # Draw output
    output = draw_detections(img, results)

    # Show
    cv2.imshow("YOLO Detection - With Custom Object", output)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()